# Plant Disease Classification Using Deep Learning and Transfer Learning

**CENG 476 - Introduction to Deep Learning**  
**Student:** Emir EVREN - **ID:** 210444038

This notebook is the **detailed, final, leakage-audited project notebook**. It is organized as an experimental story: **what I tried -> what happened -> what I changed next**. It includes the core implementation choices, the historical image-level protocol, the leakage audit, the official leaf-safe transition, the ultra-strict quarantine, final retraining, calibration, robustness, seed stability, Grad-CAM, and PlantDoc out-of-domain validation.

> **Important:** the original **99.76% ensemble** result is preserved only as a historical result that triggered the leakage audit. It is **not** the final benchmark.

### Final headline results

- **Custom CNN:** 84.62% accuracy, 0.7813 Macro-F1
- **ResNet18:** 97.66% accuracy, 0.9686 Macro-F1
- **EfficientNet-B0:** **99.01% accuracy, 0.9874 Macro-F1**
- **50/50 soft-voting ensemble:** **99.14% accuracy, 0.9897 Macro-F1**
- **3-seed EfficientNet mean:** **99.001% +/- 0.229 percentage points**
- **Mapped PlantDoc OOD:** EfficientNet **23.31%**, ensemble **25.00%**

Cached outputs shown below come from the completed runs. Heavy training is intentionally not repeated automatically when the notebook is opened.


## 0. Project Timeline

| Stage | Question | Change | What happened |
|---|---|---|---|
| 1 | Does the pipeline work? | 38-image overfit sanity check | 100% by step 40 |
| 2 | How strong is a scratch CNN? | Custom CNN + CE + AdamW + augmentation | Baseline established |
| 3 | Does transfer learning help? | ResNet18 + EfficientNet-B0 | Near-99% same-domain result |
| 4 | Can I trust 99.76%? | Exact hash + dHash + physical-leaf audit | Original split was optimistic |
| 5 | Can evaluation be safer? | Official leaf-safe split | Test fixed at 10,709 |
| 6 | Are strict near-duplicates left? | dHash<=4 review + quarantine | 39 pairs broken |
| 7 | What survives after cleaning? | Retrain/evaluate ultra-strict | Ensemble 99.14% |
| 8 | Is it a lucky seed? | Seeds 42 / 123 / 777 | 99.001 +/- 0.229 pp |
| 9 | Is there ordinary overfitting? | Clean-train vs val vs test | Severe EfficientNet overfit not supported |
| 10 | Is it robust/confident? | Calibration, bootstrap, corruptions, Grad-CAM | Same-domain strong, blur/occlusion sensitive |
| 11 | Does it generalize outside PlantVillage? | PlantDoc OOD | 23-25% |


## 1. Environment and Reproducibility

The notebook can be opened from either the repository root or the `notebooks/` directory. Analysis cells are designed to load saved artifacts instead of silently retraining models.


In [1]:
from pathlib import Path
import json
import sys
import random
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
OUTPUTS = PROJECT_ROOT / 'outputs'
AUDIT = OUTPUTS / 'audit'
FULL = AUDIT / 'full_control'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Full-control folder exists:', FULL.exists())


Project root: <repository-root>
Python: 3.14.x
PyTorch: CUDA-enabled build used for final training
CUDA available: True on the training machine
Full-control folder exists: True


### Why seed 42?

Seed 42 is not mathematically special. It is simply the predefined reference seed used for reproducibility. Later, seeds **123** and **777** are added to test whether the near-99% result depends on one lucky stochastic run.


## 2. Initial Dataset Split — What Looked Correct at First

The early pipeline used an image-level split:

- Train: **43,444**
- Validation: **5,430**
- Test: **5,431**
- Classes: **38**

Validation/test indices were disjoint, which initially looked like a clean split.


In [2]:
historical_split = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test', 'Total'],
    'Images': [43444, 5430, 5431, 54305],
})
print(historical_split.to_string(index=False))


     Split  Images
     Train   43444
Validation    5430
      Test    5431
     Total   54305


### What I learned later

A split can be index-disjoint while still being **specimen-leaky**. PlantVillage may contain multiple views of the same physical leaf, so different files can carry nearly the same biological information across train/validation/test. This realization later triggered a full leakage audit.


## 3. Data Augmentation — What I Applied

Training augmentation was designed to introduce moderate variation without destroying disease patterns. Validation and test remain deterministic.


In [3]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=InterpolationMode.BILINEAR,
        fill=(128, 128, 128),
    ),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

evaluation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Training: RandomResizedCrop(224), HFlip(0.5), Rotation(+/-15), ColorJitter(0.2), ImageNet normalization')
print('Validation/Test: Resize(256) -> CenterCrop(224) -> ImageNet normalization')


Training: RandomResizedCrop(224), HFlip(0.5), Rotation(+/-15), ColorJitter(0.2), ImageNet normalization
Validation/Test: Resize(256) -> CenterCrop(224) -> ImageNet normalization


### Why I did not augment validation/test

Validation and test metrics must be comparable across epochs and across models. Random augmentation would make the same image change every evaluation pass, adding unnecessary measurement noise.


## 4. Custom CNN — My From-Scratch Baseline

The baseline is intentionally compact and transparent. It starts from random initialization and has four convolutional blocks.


In [4]:
from torch import nn

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=38, dropout_rate=0.4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

baseline = BaselineCNN()
print('Output classes:', baseline.classifier[-1].out_features)
print('Trainable parameters:', f'{sum(p.numel() for p in baseline.parameters()):,}')


Output classes: 38
Trainable parameters: 399,142


## 5. Loss Function — What I Used and Why

### Final choice

```python
criterion = nn.CrossEntropyLoss()
```

For one sample:

\[
L = -\log\left(\frac{e^{z_y}}{\sum_{j=1}^{38} e^{z_j}}\right)
\]

where `z_j` are the raw logits and `y` is the correct class.

### Why there is no Softmax before the loss

`CrossEntropyLoss` expects **raw logits** and internally performs the stable equivalent of LogSoftmax + Negative Log Likelihood.

Correct:

```python
logits = model(images)
loss = criterion(logits, labels)
```

Not used:

```python
probs = softmax(logits)
loss = criterion(probs, labels)
```

Softmax is only needed later when I need actual probabilities for ensemble voting, calibration, confidence analysis, and ROC-related work.


In [5]:
criterion = nn.CrossEntropyLoss()
print('Loss function: CrossEntropyLoss')
print('Input to loss: raw logits [batch, 38]')
print('Softmax before CE: NO')


Loss function: CrossEntropyLoss
Input to loss: raw logits [batch, 38]
Softmax before CE: NO


### What happened after using this setup

In the tiny sanity-overfit test, the pipeline reached **100% training accuracy by step 40**. This gave evidence that the model, labels, loss, gradients, and optimizer were connected correctly.


## 6. Gradient Descent and AdamW — How Parameters Were Updated

Core idea:

\[
w_{new}=w_{old}-\eta \frac{\partial L}{\partial w}
\]

The gradient tells how the loss changes with each parameter. We move in the negative-gradient direction to reduce loss.

I used **AdamW** with:
- betas = `(0.9, 0.999)`
- weight decay = `1e-4`


In [6]:
from torch.optim import AdamW

dummy_parameter = nn.Parameter(torch.zeros(1))
optimizer_example = AdamW(
    [dummy_parameter],
    lr=5e-4,
    betas=(0.9, 0.999),
    weight_decay=1e-4,
)
print('Optimizer: AdamW')
print('betas:', optimizer_example.defaults['betas'])
print('weight_decay:', optimizer_example.defaults['weight_decay'])


Optimizer: AdamW
betas: (0.9, 0.999)
weight_decay: 0.0001


### Why AdamW instead of plain Adam

AdamW decouples weight decay from the adaptive gradient update. This makes regularization behavior cleaner than treating L2 regularization as part of Adam's adaptive gradient calculation.


## 7. Training Step — Exact Order Used

The order matters: zero old gradients -> forward pass -> compute loss -> backward -> optimizer step. Mixed precision is used on CUDA.


In [7]:
def train_one_batch(model, images, labels, criterion, optimizer, scaler, device):
    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == 'cuda'),
    ):
        logits = model(images)
        loss = criterion(logits, labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    predictions = logits.argmax(dim=1)
    return loss, predictions


### Why zero_grad?

PyTorch accumulates gradients by default. If old gradients are not cleared, gradients from previous batches are added to the current batch and the intended mini-batch update is changed.


## 8. Sanity Check — Deliberately Overfit 38 Images

Before long training, I tested whether the network could memorize a tiny set.

Settings:
- 38 images
- augmentation off
- dropout 0
- weight decay 0
- AMP enabled

Observed:
- step 30: **71.05%**
- step 40: **100.00%**


In [8]:
sanity = pd.DataFrame([
    [30, 71.05],
    [40, 100.00],
], columns=['Step', 'Accuracy (%)'])
print(sanity.to_string(index=False))
print('Sanity check: PASS')


 Step  Accuracy (%)
   30         71.05
   40        100.00
Sanity check: PASS


### Decision after this result

Because the network could memorize the tiny set, I had evidence that the forward pass, labels, CrossEntropyLoss, backward pass, and optimizer update were functioning. I then moved to longer experiments where the goal is **generalization**, not memorization.


## 9. Learning-Rate Pilot — Why the Baseline Settled on 5e-4

Short baseline pilots were run around:
- `1e-3`
- `5e-4`
- `3e-4`

A recorded `1e-3` pilot after epoch 1 gave:
- train accuracy: **61.27%**
- validation accuracy: **45.47%**
- validation Macro-F1: **0.3525**

The final full baseline used `5e-4` as the more conservative longer-run learning rate.


In [9]:
# Historical pilot commands used different learning rates around:
pilot_lrs = [1e-3, 5e-4, 3e-4]
pilot_lrs


## 10. Dropout Ablation — How Regularization Strength Changed Learning

A controlled short pilot changed dropout while keeping the rest of the setup fixed.


In [10]:
dropout = pd.DataFrame([
    [0.20, 0.4990],
    [0.40, 0.4953],
    [0.60, 0.4311],
], columns=['Dropout', 'Best Val Macro-F1'])
print(dropout.to_string(index=False))


 Dropout  Best Val Macro-F1
     0.2             0.4990
     0.4             0.4953
     0.6             0.4311


Interpretation: `0.60` was clearly too aggressive for the short baseline pilot and slowed learning. The final baseline remained at **0.40**. The pilot did not use locked-test performance to retroactively select dropout.


## 11. Transfer Learning — The Main Modeling Upgrade

The scratch CNN provides a transparent baseline, but it must learn all visual features from the project dataset. I therefore added two ImageNet-pretrained models and fine-tuned them end-to-end.


In [11]:
from torchvision.models import (
    ResNet18_Weights, resnet18,
    EfficientNet_B0_Weights, efficientnet_b0,
)

def make_resnet18(num_classes=38, dropout=0.30):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes),
    )
    return model

def make_efficientnet(num_classes=38, dropout=0.30):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

model_sizes = pd.DataFrame([
    ['Custom CNN', 399_142],
    ['ResNet18', 11_196_006],
    ['EfficientNet-B0', 4_056_226],
], columns=['Model', 'Trainable parameters'])
print(model_sizes.to_string(index=False))


          Model  Trainable parameters
     Custom CNN               399142
       ResNet18             11196006
EfficientNet-B0              4056226


### What changed here

- Baseline initialization: random
- ResNet18/EfficientNet initialization: ImageNet weights
- Baseline dropout: 0.40
- Transfer classifier dropout: 0.30
- Transfer models: **full fine-tuning**, not frozen feature extraction


## 12. Differential Learning Rates for Transfer Learning

The pretrained backbone already contains useful visual representations, so I update it conservatively. The new classifier starts randomly and can adapt faster.


In [12]:
model = make_efficientnet()
backbone_parameters = []
classifier_parameters = []

for name, parameter in model.named_parameters():
    if name.startswith('classifier.'):
        classifier_parameters.append(parameter)
    else:
        backbone_parameters.append(parameter)

optimizer = AdamW([
    {'params': backbone_parameters, 'lr': 1e-4, 'name': 'backbone'},
    {'params': classifier_parameters, 'lr': 5e-4, 'name': 'classifier'},
], betas=(0.9, 0.999), weight_decay=1e-4)

print('Backbone LR:', optimizer.param_groups[0]['lr'])
print('Classifier LR:', optimizer.param_groups[1]['lr'])
print('Ratio: classifier updates use 5x LR')


Backbone LR: 0.0001
Classifier LR: 0.0005
Ratio: classifier updates use 5x LR


## 13. Scheduler, Checkpoint Selection, and Early Stopping

These mechanisms solve different problems:

- **ReduceLROnPlateau:** reacts to validation-loss plateaus
- **Checkpoint selection:** chooses the best validation Macro-F1
- **Tie-break:** lower validation loss
- **Early stopping safeguard:** stops after repeated non-improvement

Important: the selected final runs reached their configured maximum epoch count before early stopping terminated them.


In [13]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)
print('Scheduler monitor: validation loss')
print('Scheduler factor: 0.5')
print('Scheduler patience: 2')
print('Minimum LR: 1e-06')
print('Checkpoint metric: validation Macro-F1')


Scheduler monitor: validation loss
Scheduler factor: 0.5
Scheduler patience: 2
Minimum LR: 1e-06
Checkpoint metric: validation Macro-F1


## 14. Metrics — Why I Did Not Use Accuracy Alone

Accuracy can be dominated by large classes. Macro-F1 computes F1 separately for all 38 classes and gives each class equal weight. This is why **validation Macro-F1** is the main checkpoint-selection metric.

For each class:

\[Precision=TP/(TP+FP)\]

\[Recall=TP/(TP+FN)\]

\[F1=2PR/(P+R)\]


## 15. Historical First Benchmark — The Result That Triggered the Audit

The early image-level protocol produced:

| Model | Historical test accuracy |
|---|---:|
| Custom CNN | 87.42% |
| ResNet18 | 99.26% |
| EfficientNet-B0 | 99.52% |
| Soft-voting ensemble | **99.76%** |

Instead of treating 99.76% as the end of the project, I treated it as a reason to test whether the split was too easy or leaky.


In [14]:
historical = pd.DataFrame([
    ['Custom CNN', 87.42],
    ['ResNet18', 99.26],
    ['EfficientNet-B0', 99.52],
    ['50/50 Ensemble', 99.76],
], columns=['Model', 'Historical Accuracy (%)'])
print(historical.to_string(index=False))


          Model  Historical Accuracy (%)
     Custom CNN                    87.42
       ResNet18                    99.26
EfficientNet-B0                    99.52
  50/50 Ensemble                    99.76


## 16. Leakage Audit — Exact Duplicates, Near-Duplicates, and Same Physical Leaf

I checked the original split at three levels:

### A. Exact image hashing
Find byte/content-identical images across splits.

### B. Perceptual dHash
Find visually near-identical images even when encoded differently.

### C. Physical-leaf mapping
Where metadata could identify a leaf, check whether multiple photographs of the same leaf cross train/validation/test boundaries.

### Original audit result
- exact cross-split duplicate groups/pairs: **10**
- near-duplicate dHash<=4 pairs: **68**
- mapped same-physical-leaf cross-split groups: **4,956**

This means the first split was not strong enough for the final claim.


In [15]:
audit = pd.DataFrame([
    ['Exact cross-split', 10],
    ['dHash<=4 near-duplicate pairs', 68],
    ['Mapped same-physical-leaf cross-split groups', 4956],
], columns=['Audit check', 'Original'])
print(audit.to_string(index=False))


                                  Audit check  Original
                       Exact cross-split           10
             dHash<=4 near-duplicate pairs           68
Mapped same-physical-leaf cross-split groups         4956


### Important: what I do NOT conclude

I do **not** claim that all of the historical 99.76% was caused by leakage. The later ultra-strict ensemble still reaches 99.14%, so the controlled-domain task remains genuinely easy for transfer-learning models.


## 17. Official Leaf-Safe Protocol — First Cleaning Stage

I switched to the official PlantVillage split boundary.

Key changes:
1. keep official test unchanged at **10,709**
2. remove **5 byte-identical train/test collisions from train**
3. create internal train/validation from the training side
4. respect mapped leaf identity

Intermediate leaf-safe split:
- Train: **39,125**
- Validation: **4,466**
- Test: **10,709**

Checks:
- exact train/test hash overlap: **0**
- mapped leaf train/test overlap: **0**
- mapped leaf train/validation overlap: **0**


In [16]:
leafsafe = pd.DataFrame([
    ['Official leaf-safe', 39125, 4466, 10709],
], columns=['Stage', 'Train', 'Validation', 'Test'])
print(leafsafe.to_string(index=False))


                 Stage  Train  Validation  Test
Official leaf-safe   39125        4466 10709


## 18. Ultra-Strict dHash Quarantine — Final Cleaning Stage

A strict dHash<=4 review still found **39** cross-split pairs:
- test-train: 27
- test-validation: 4
- train-validation: 8

Because some unmapped pairs looked like the same physical specimen during manual review, I used a conservative deterministic rule:

> break every strict pair by removing the lower-priority side with priority **test > validation > train**

The test set itself stays unchanged.


In [17]:
strict_review = pd.DataFrame([
    ['test-train', 27],
    ['test-validation', 4],
    ['train-validation', 8],
    ['total', 39],
], columns=['Pair type', 'Count'])
print(strict_review.to_string(index=False))


       Pair type  Count
      test-train     27
test-validation      4
train-validation      8
           total     39


### Quarantine result

- Train removed: **34**
- Validation removed: **4**
- Test removed: **0**
- Remaining strict dHash<=4 cross-split pairs: **0**

Final ultra-strict split:
- Train: **39,091**
- Validation: **4,462**
- Test: **10,709**
- Total: **54,262**


In [18]:
ultra = pd.DataFrame([
    ['Train', 39091, 34],
    ['Validation', 4462, 4],
    ['Test', 10709, 0],
    ['Total', 54262, 38],
], columns=['Split', 'Final images', 'Quarantined'])
print(ultra.to_string(index=False))


     Split  Final images  Quarantined
     Train         39091           34
Validation          4462            4
      Test         10709            0
     Total         54262           38


### Exact test-boundary wording

> **The locked test set was never modified and was never used for model training, checkpoint selection, hyperparameter tuning, or ensemble-weight selection. Test images were used only for a deterministic, model-independent duplicate and near-duplicate integrity audit.**

This is more accurate than saying that the test was never 'seen'.


## 19. Retraining Under the Final Protocol

After changing the split, I did not reuse the historical scores. The models were retrained/evaluated under the ultra-strict manifest.

| Setting | Custom CNN | ResNet18 | EfficientNet-B0 |
|---|---:|---:|---:|
| Initialization | Random | ImageNet | ImageNet |
| Batch size | 64 | 32 | 32 |
| Max epochs | 15 | 12 | 12 |
| Backbone LR | 5e-4 | 1e-4 | 1e-4 |
| Classifier LR | 5e-4 | 5e-4 | 5e-4 |
| Dropout | 0.40 | 0.30 | 0.30 |
| Weight decay | 1e-4 | 1e-4 | 1e-4 |
| Loss | CE | CE | CE |
| Checkpoint metric | Val Macro-F1 | Val Macro-F1 | Val Macro-F1 |


## 20. Historical vs Final Results

| Model | Historical image-level | Final ultra-strict | Change |
|---|---:|---:|---:|
| Custom CNN | 87.42% | **84.62%** | -2.80 pp |
| ResNet18 | 99.26% | **97.66%** | -1.60 pp |
| EfficientNet-B0 | 99.52% | **99.01%** | -0.51 pp |
| Ensemble | 99.76% | **99.14%** | -0.62 pp |

These differences are **not a pure causal leakage estimate**, because the split/protocol changed. The main point is that the very high transfer-learning result survives a much stricter evaluation.


In [19]:
final_results = pd.DataFrame([
    ['Custom CNN', 84.620413, 0.781323, 0.836106, 1647],
    ['ResNet18', 97.656177, 0.968603, 0.976314, 251],
    ['EfficientNet-B0', 99.010178, 0.987373, 0.990114, 106],
    ['50/50 Ensemble', 99.140910, 0.989733, 0.991396, 92],
], columns=['Model','Accuracy (%)','Macro-F1','Weighted-F1','Errors'])
print(final_results.to_string(index=False))


          Model  Accuracy (%)  Macro-F1  Weighted-F1  Errors
     Custom CNN     84.620413  0.781323     0.836106    1647
       ResNet18     97.656177  0.968603     0.976314     251
EfficientNet-B0     99.010178  0.987373     0.990114     106
  50/50 Ensemble     99.140910  0.989733     0.991396      92


### Main modeling conclusion

Transfer learning dramatically outperforms the from-scratch baseline. EfficientNet-B0 is the strongest individual model and also has far fewer parameters than ResNet18.


## 21. Ensemble — How I Selected 50/50 Without Using Test

Soft voting:

\[
p_{ens} = w_R p_{ResNet18} + w_E p_{EfficientNet}
\]

Candidate weights were compared on **validation only**.


In [20]:
ensemble_weights = pd.DataFrame([
    [1.00,0.00,0.988122,0.986237],
    [0.75,0.25,0.990587,0.988722],
    [0.50,0.50,0.993725,0.992033],
    [0.25,0.75,0.989691,0.986400],
    [0.00,1.00,0.986777,0.982431],
], columns=['ResNet weight','EfficientNet weight','Val Accuracy','Val Macro-F1'])
print(ensemble_weights.to_string(index=False))
print('Selected: 50/50 using validation only')


ResNet weight  EfficientNet weight  Val Accuracy  Val Macro-F1
         1.00                 0.00      0.988122      0.986237
         0.75                 0.25      0.990587      0.988722
         0.50                 0.50      0.993725      0.992033
         0.25                 0.75      0.989691      0.986400
         0.00                 1.00      0.986777      0.982431
Selected: 50/50 using validation only


### What happened

The 50/50 combination had the highest validation Macro-F1 (**0.992033**), so it was fixed before locked-test evaluation.


## 22. Generalization Gap — Did EfficientNet Simply Memorize Training?

A deterministic balanced clean-training subset was evaluated:
- 20 images/class
- 38 classes
- **760 images total**

Selected-checkpoint accuracies:

| Model | Clean train | Validation | Test |
|---|---:|---:|---:|
| Baseline | 77.89% | 83.55% | 84.62% |
| ResNet18 | 98.16% | 98.81% | 97.66% |
| EfficientNet-B0 | 99.74% | 98.68% | 99.01% |


In [21]:
gaps = pd.DataFrame([
    ['Baseline',77.8947,83.55,84.620413],
    ['ResNet18',98.1579,98.8122,97.656177],
    ['EfficientNet-B0',99.7368,98.6777,99.010178],
], columns=['Model','Clean train','Validation','Test'])
print(gaps.to_string(index=False))


          Model  Clean train  Validation      Test
       Baseline    77.894700   83.550000 84.620413
       ResNet18    98.157900   98.812200 97.656177
EfficientNet-B0    99.736800   98.677700 99.010178


### Interpretation

EfficientNet's clean-train/validation gap is about one percentage point and test is slightly above validation. This does **not** support severe conventional train overfitting as the main explanation for its near-99% PlantVillage result.

This does **not** imply good field-domain generalization; PlantDoc later shows the opposite.


## 23. Calibration — Is 99% Confidence Meaningful?

I added:
- ECE (Expected Calibration Error)
- NLL (Negative Log Likelihood)
- multiclass Brier score

Final values:

| Model | ECE (15 bins) | NLL | Brier |
|---|---:|---:|---:|
| EfficientNet-B0 | **0.003845** | 0.035704 | 0.016145 |
| Ensemble | 0.009121 | 0.035419 | 0.017435 |

Low ECE means predicted confidence and observed correctness are closely aligned **inside this same-domain test**.


In [22]:
calibration = pd.DataFrame([
    ['EfficientNet-B0',0.003845,0.035704,0.016145],
    ['Ensemble',0.009121,0.035419,0.017435],
], columns=['Model','ECE','NLL','Brier'])
print(calibration.to_string(index=False))


          Model      ECE      NLL    Brier
EfficientNet-B0 0.003845 0.035704 0.016145
       Ensemble 0.009121 0.035419 0.017435


## 24. Bootstrap Confidence Intervals — Uncertainty Around the Score

I used **1,000 ordinary bootstrap samples** from the locked test predictions.

EfficientNet:
- accuracy 95% CI approx **98.81% to 99.20%**
- Macro-F1 95% CI approx **0.9847 to 0.9900**

Ensemble:
- accuracy 95% CI approx **98.95% to 99.31%**
- Macro-F1 95% CI approx **0.9873 to 0.9918**

This makes uncertainty visible instead of treating 99.01 as an exact universal constant.


## 25. Random-Label Sanity — What If Labels Carry No Real Signal?

I deliberately shuffled the training labels.

Setup:
- balanced train subset = 20/class = **760**
- validation = 10/class = **380**
- frozen pretrained EfficientNet backbone
- classifier trained for 5 epochs
- validation still uses the **true labels**

Chance for 38 classes: `1 / 38 approx 2.63%`

Observed true-label validation:
- accuracy = **1.8421%**
- Macro-F1 = **0.01832**

**PASS**


In [23]:
chance = 100/38
print(f'Chance accuracy: {chance:.4f}%')
print('True-label validation accuracy after shuffled-label training: 1.8421%')
print('Validation Macro-F1: 0.01832')
print('Result: PASS')


Chance accuracy: 2.6316%
True-label validation accuracy after shuffled-label training: 1.8421%
Validation Macro-F1: 0.01832
Result: PASS


### What this means

The negative control does not prove mathematically that every possible leak is absent, but it is strong evidence that the classifier cannot obtain meaningful true-label validation performance after label information is deliberately destroyed.


## 26. Robustness Stress Tests — Which Changes Hurt?

Post-hoc test accuracy:

| Stress | EfficientNet | Ensemble |
|---|---:|---:|
| Clean | 99.01% | 99.14% |
| Brightness 0.60 | 98.83% | 99.18% |
| Brightness 1.40 | 98.07% | 98.73% |
| Contrast 0.60 | 98.49% | 98.79% |
| JPEG q=30 | 98.13% | 98.72% |
| Rotation 15 deg | 99.41% | 99.51% |
| Gaussian blur r=2 | **84.08%** | **86.53%** |
| Center occluded 60% | **55.38%** | **64.45%** |
| Border occluded, keep center 60% | **58.85%** | **77.93%** |


In [24]:
robustness = pd.DataFrame([
    ['clean',99.0102,99.1409],
    ['brightness0.60',98.8328,99.1783],
    ['brightness1.40',98.0670,98.7300],
    ['contrast0.60',98.4873,98.7861],
    ['JPEG30',98.1324,98.7207],
    ['rotation15',99.4117,99.5051],
    ['Gaussian blur r2',84.0788,86.5254],
    ['center occluded 60',55.3833,64.4505],
    ['border occluded keep center',58.8477,77.9344],
], columns=['Stress','EfficientNet','Ensemble'])
print(robustness.to_string(index=False))


                     Stress  EfficientNet  Ensemble
                      clean      99.0102   99.1409
             brightness0.60      98.8328   99.1783
             brightness1.40      98.0670   98.7300
                contrast0.60      98.4873   98.7861
                     JPEG30      98.1324   98.7207
                 rotation15      99.4117   99.5051
            Gaussian blur r2      84.0788   86.5254
         center occluded 60      55.3833   64.4505
border occluded keep center      58.8477   77.9344


### Observation -> interpretation

The model is fairly stable under moderate brightness, contrast, JPEG compression, and rotation. Blur and large occlusion are much more damaging. This suggests sensitivity to fine visual detail, but the stress tests do **not** prove one specific causal shortcut.


## 27. Grad-CAM — What the Model Appears to Attend To

I generated Grad-CAM contact sheets for:
- high-confidence correct cases
- error cases

The maps often overlap visible symptom regions, but Grad-CAM is used only as **qualitative supporting evidence**, not as causal proof.

When the generated figures are present in `outputs/figures/full_control/`, the following cells can display them locally.


In [25]:
from IPython.display import display, Image

for filename in [
    'efficientnet_gradcam_correct.jpg',
    'efficientnet_gradcam_errors.jpg',
]:
    path = OUTPUTS / 'figures' / 'full_control' / filename
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Local figure not found:', filename)


## 28. Seed Stability — Is the 99% Result a Lucky Run?

EfficientNet-B0 was retrained on the same ultra-strict manifest and same hyperparameters with three seeds.

| Seed | Val Macro-F1 | Test Accuracy | Test Macro-F1 | Errors |
|---:|---:|---:|---:|---:|
| 42 | 0.982431 | 99.010% | 0.987373 | 106 |
| 123 | 0.986704 | 98.767% | 0.983668 | 132 |
| 777 | 0.992056 | 99.225% | 0.990149 | 83 |

Summary:
- mean = **99.001%**
- standard deviation = **0.229 pp**
- range = **0.458 pp**


In [26]:
seed_runs = pd.DataFrame([
    [42,0.982431,99.010,0.987373,106],
    [123,0.986704,98.767,0.983668,132],
    [777,0.992056,99.225,0.990149,83],
], columns=['Seed','Val Macro-F1','Test Accuracy','Test Macro-F1','Errors'])
print(seed_runs.to_string(index=False))
print('Mean accuracy: 99.001%')
print('Std: 0.229 pp')


 Seed  Val Macro-F1  Test Accuracy  Test Macro-F1  Errors
   42      0.982431         99.010       0.987373     106
  123      0.986704         98.767       0.983668     132
  777      0.992056         99.225       0.990149      83
Mean accuracy: 99.001%
Std: 0.229 pp


### Why I do not report seed 777 as the final benchmark

Seed 777 has the highest observed test accuracy, but selecting the best seed **after seeing test results** would be cherry-picking. I keep the predefined seed-42 benchmark and summarize all seeds together.


## 29. PlantDoc OOD — The Most Important Limitation

To test domain shift, I evaluated a manually mapped PlantDoc subset **without retraining**:

- **236 images**
- **27 source classes**
- mapped to compatible PlantVillage labels

Results:
- EfficientNet-B0: **23.31%**, mapped Macro-F1 **0.2183**
- Ensemble: **25.00%**, mapped Macro-F1 **0.2349**


In [27]:
plantdoc = pd.DataFrame([
    ['EfficientNet-B0',23.31,0.2183],
    ['50/50 Ensemble',25.00,0.2349],
], columns=['Model','OOD Accuracy (%)','Mapped Macro-F1'])
print(plantdoc.to_string(index=False))


          Model  OOD Accuracy (%)  Mapped Macro-F1
EfficientNet-B0             23.31           0.2183
  50/50 Ensemble             25.00           0.2349


### How this changed my conclusion

Before OOD evaluation, near-99% could be misread as a deployment-level result. PlantDoc shows that the correct conclusion is much narrower:

> **near-99% is reproducible inside controlled PlantVillage, but not validated for real-field imagery.**


## 30. Full Before -> Change -> After Table

| Question / problem | Before | What I changed | Result after change |
|---|---|---|---|
| Does the code learn? | Unknown | 38-image overfit sanity test | 100% at step 40 |
| Multiclass loss | Needed correct formulation | Raw logits + CrossEntropyLoss | Stable training |
| Optimization | Needed regularized updates | AdamW + wd=1e-4 | Used in final runs |
| Learning rate | Uncertain | 1e-3 / 5e-4 / 3e-4 pilots | baseline full run at 5e-4 |
| Dropout | Uncertain | 0.2 / 0.4 / 0.6 ablation | 0.6 too aggressive in short pilot |
| Scratch model capacity | Limited | Add pretrained ResNet18/EfficientNet | near-99% same-domain |
| Original 99.76% credibility | Suspiciously high | Exact/dHash/leaf audit | leakage-like overlap confirmed |
| Split protocol | image-level | official leaf-safe split | test=10,709, mapped overlap=0 |
| Residual near-duplicates | 39 strict pairs | deterministic quarantine | 0 dHash<=4 pairs remain |
| Final benchmark | historical 99.76 | retrain/evaluate ultra-strict | ensemble=99.14 |
| Lucky seed concern | one seed | 42/123/777 | 99.001 +/- 0.229 pp |
| Overfitting concern | accuracy only | clean-train/val/test gaps | severe EfficientNet overfit not supported |
| Confidence uncertainty | accuracy only | ECE/NLL/Brier + bootstrap | ECE=0.003845; tight CI |
| Robustness unknown | clean only | corruptions + occlusions | blur/occlusion sensitivity |
| Interpretability unknown | predictions only | Grad-CAM | supportive qualitative evidence |
| External generalization unknown | PlantVillage only | PlantDoc OOD | 23-25%; strong domain dependence |


## 31. Final Recommendation

If one individual model must be chosen, **EfficientNet-B0** is the strongest project model: it has fewer parameters than ResNet18 and reaches 99.01% on the final PlantVillage test.

If the objective is maximum same-domain predictive performance, the validation-selected **50/50 ensemble** reaches 99.14%.

Neither result should be presented as field-deployment accuracy.


## 32. What I Can Claim vs What I Cannot Claim

### Supported
- the first image-level protocol was optimistic/leaky
- final exact cross-split duplicates detected by our checks = 0
- final mapped physical-leaf overlap detected = 0
- final strict dHash<=4 cross-split pairs detected = 0
- EfficientNet = 99.01% and ensemble = 99.14% on final PlantVillage protocol
- 3-seed EfficientNet mean = 99.001 +/- 0.229 pp
- PlantDoc OOD performance is much lower

### Not supported
- mathematical proof that every unmapped specimen is unique
- claim that all historical 99.76% was caused by leakage
- claim that Grad-CAM proves causal disease-feature use
- claim of near-99% real-world field accuracy


## 33. Implementation Details — Exact Pieces of Code Behind the Decisions

This section collects the **implementation-level details** behind the methodological decisions above. The goal is to make the notebook answer questions such as: *What exactly did I code? What signal did I monitor? What changed after the result? Which decisions used validation and which did not?*


### 33.1 Initial image-level split code — why it looked safe but was not enough

At the beginning, the validation directory was split with a deterministic stratified split. This guaranteed that **the exact same index** could not appear in both validation and test, but it did **not** guarantee that two different image files depicting the same physical leaf could not cross the boundary.


In [28]:
from sklearn.model_selection import train_test_split

# Simplified form of the historical split logic
all_indices = list(range(100))
targets = [i % 10 for i in range(100)]
validation_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.5,
    random_state=42,
    stratify=targets,
)
print('Index overlap between validation/test:', len(set(validation_indices) & set(test_indices)))
print('Important later finding: index-level separation != specimen-level separation')


Index overlap between validation/test: 0
Important later finding: index-level separation != specimen-level separation


**Decision after audit:** I stopped treating simple index disjointness as sufficient evidence of independence. The final protocol therefore adds **content hashing, perceptual hashing, and physical-leaf grouping**.


### 33.2 Exact duplicate detection

For exact-content duplicates, images are represented by a cryptographic hash. If the same digest appears in two splits, the content is identical even if the filenames differ.


In [29]:
import hashlib

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print('Original audit: 10 exact cross-split duplicate groups/pairs')
print('Final ultra-strict audit: 0 detected exact cross-split overlap')


Original audit: 10 exact cross-split duplicate groups/pairs
Final ultra-strict audit: 0 detected exact cross-split overlap


### 33.3 Perceptual dHash — why exact hashing alone is insufficient

Two files can be visually almost identical but have different bytes because of JPEG recompression, resizing, metadata, or minor pixel differences. For this reason I also used a **difference hash (dHash)** and compared Hamming distance.


In [30]:
from PIL import Image

def dhash(image, hash_size=8):
    gray = image.convert('L').resize((hash_size + 1, hash_size))
    pixels = list(gray.getdata())
    value = 0
    for row in range(hash_size):
        for col in range(hash_size):
            left = pixels[row * (hash_size + 1) + col]
            right = pixels[row * (hash_size + 1) + col + 1]
            value = (value << 1) | int(left > right)
    return value

def hamming_distance(a, b):
    return (a ^ b).bit_count()

print('Original image-level audit near-duplicate pairs: 68')
print('Final strict review before quarantine: 39')
print('Remaining after ultra-strict quarantine: 0')


Original image-level audit near-duplicate pairs: 68
Final strict review before quarantine: 39
Remaining after ultra-strict quarantine: 0


### 33.4 Ultra-strict quarantine rule

I did not change the official test set. For every remaining strict cross-split pair I quarantined the **lower-priority** side using `test > validation > train`.


In [31]:
priority = {'train': 0, 'validation': 1, 'test': 2}

def lower_priority_side(split_a, split_b):
    if priority[split_a] < priority[split_b]:
        return 'a'
    if priority[split_b] < priority[split_a]:
        return 'b'
    return None

print('Strict pairs before: 39')
print('Quarantined: train=34, validation=4, test=0')
print('Strict pairs after: 0')


Strict pairs before: 39
Quarantined: train=34, validation=4, test=0
Strict pairs after: 0


### 33.5 Final training transform versus evaluation transform

A critical implementation detail is that stochastic operations exist **only in the training pipeline**. Validation and test are deterministic so model comparison is repeatable.


In [32]:
print('Train: stochastic augmentation ON')
print('Validation: deterministic')
print('Test: deterministic')


Train: stochastic augmentation ON
Validation: deterministic
Test: deterministic


### 33.6 Forward pass, loss, gradient, and optimizer — exact order

This is the core learning sequence. A common mistake is to apply Softmax before `CrossEntropyLoss`; I did not do that.


In [33]:
# Core sequence used in training:
# optimizer.zero_grad(set_to_none=True)
# logits = model(images)
# loss = criterion(logits, labels)
# scaler.scale(loss).backward()
# scaler.step(optimizer)
# scaler.update()
# predictions = logits.argmax(dim=1)

print('Loss receives raw logits')
print('Backward computes gradients')
print('AdamW updates parameters')
print('Argmax converts logits to predicted class indices')


Loss receives raw logits
Backward computes gradients
AdamW updates parameters
Argmax converts logits to predicted class indices


The underlying gradient-descent idea is `w_(t+1) = w_t - eta * grad(L)`. AdamW maintains adaptive first/second moment estimates, but the purpose remains the same: update parameters in a direction that reduces loss.


### 33.7 Why `optimizer.zero_grad()` is necessary

PyTorch accumulates gradients by default. Without clearing them, the next batch would add its gradients to the previous batch's gradients.


In [34]:
print('Gradients are reset once per mini-batch before backward().')


Gradients are reset once per mini-batch before backward().


### 33.8 Differential learning rates — exact optimizer groups

The pretrained backbone and the new classifier do not start from the same knowledge state. Therefore I used a smaller learning rate for pretrained parameters and a larger learning rate for the newly initialized classifier.


In [35]:
print('Backbone LR:   1.00e-04')
print('Classifier LR: 5.00e-04')
print('Weight decay:  1.00e-04')


Backbone LR:   1.00e-04
Classifier LR: 5.00e-04
Weight decay:  1.00e-04


### 33.9 Scheduler — what changed when validation loss plateaued

`ReduceLROnPlateau` uses validation **loss**, not test loss and not test accuracy.


In [36]:
print('Monitor: validation loss')
print('Factor: 0.5')
print('Patience: 2')
print('Minimum LR: 1e-6')


Monitor: validation loss
Factor: 0.5
Patience: 2
Minimum LR: 1e-6


### 33.10 Checkpoint selection — Macro-F1 first, validation loss as tie-break

The scheduler and checkpoint serve different purposes: scheduler -> smooth optimization signal (`validation_loss`); best model -> balanced classification signal (`validation_macro_f1`).


In [37]:
minimum_improvement = 1e-4
# f1_improved = current_macro_f1 > best_validation_macro_f1 + minimum_improvement
# f1_tied = abs(current_macro_f1 - best_validation_macro_f1) <= minimum_improvement
# loss_improved_on_tie = f1_tied and current_validation_loss < best_validation_loss
print('Primary checkpoint criterion: validation Macro-F1')
print('Tie-break: lower validation loss')
print('Test metrics: not used')


Primary checkpoint criterion: validation Macro-F1
Tie-break: lower validation loss
Test metrics: not used


### 33.11 Early stopping — implemented versus actually triggered

The code tracks the number of epochs without validation-Macro-F1 improvement. However, the selected final runs reached their configured epoch limit before the patience condition terminated training.


In [38]:
print('Early-stopping logic: implemented')
print('Final selected runs: maximum configured epoch reached first')


Early-stopping logic: implemented
Final selected runs: maximum configured epoch reached first


### 33.12 Metric implementation — why Macro-F1 matters

Accuracy counts total correct predictions. Macro-F1 calculates F1 for each class separately and averages all 38 values equally.


In [39]:
from sklearn.metrics import f1_score
print('All 38 classes contribute equally to Macro-F1.')


All 38 classes contribute equally to Macro-F1.


For class c: `Precision = TP/(TP+FP)`, `Recall = TP/(TP+FN)`, and `F1 = 2*P*R/(P+R)`. Macro-F1 is the unweighted mean of the 38 class F1 scores.


### 33.13 Soft voting — exact ensemble equation and implementation

The selected ensemble averages the **probability distributions**, not the predicted class labels.


In [40]:
# p_resnet = torch.softmax(resnet_logits, dim=1)
# p_effnet = torch.softmax(efficientnet_logits, dim=1)
# p_ensemble = 0.5 * p_resnet + 0.5 * p_effnet
# ensemble_pred = p_ensemble.argmax(dim=1)
print('Validation-selected weight: 0.50 ResNet18 + 0.50 EfficientNet-B0')
print('Final test accuracy: 99.140910%')
print('Final test Macro-F1: 0.989733')


Validation-selected weight: 0.50 ResNet18 + 0.50 EfficientNet-B0
Final test accuracy: 99.140910%
Final test Macro-F1: 0.989733


### 33.14 Why ensemble selection was validation-only

Five candidate weight combinations were compared before locked-test evaluation: 100/0, 75/25, 50/50, 25/75, and 0/100. The best validation Macro-F1 was 0.992033 at 50/50.


In [41]:
print('Best validation Macro-F1: 50/50 -> 0.992033')
print('Test-set weight search: NOT performed')


Best validation Macro-F1: 50/50 -> 0.992033
Test-set weight search: NOT performed


### 33.15 Calibration — ECE implementation idea

ECE checks whether confidence is aligned with actual correctness. Predictions with confidence around 0.90 should ideally be correct around 90% of the time.


In [42]:
def expected_calibration_error(confidence, correct, n_bins=15):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if not mask.any():
            continue
        bin_conf = confidence[mask].mean()
        bin_acc = correct[mask].mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)
    return float(ece)

print('EfficientNet ECE (15 bins): 0.003845')
print('Ensemble ECE (15 bins):     0.009121')


EfficientNet ECE (15 bins): 0.003845
Ensemble ECE (15 bins):     0.009121


### 33.16 Random-label sanity — exact experimental logic

This is intentionally a **negative control**. If the pipeline still achieved high true-label validation performance after training labels were destroyed, that would be suspicious.


In [43]:
print('38-class chance accuracy: 2.63%')
print('Observed true-label validation accuracy: 1.8421%')
print('Observed validation Macro-F1: 0.01832')
print('Sanity result: PASS')


38-class chance accuracy: 2.63%
Observed true-label validation accuracy: 1.8421%
Observed validation Macro-F1: 0.01832
Sanity result: PASS


### 33.17 Seed experiment — what was allowed to change

The **manifest and hyperparameters remained fixed**. Only stochastic training state changed with the seed.


In [44]:
print('Seed 42:  99.010%')
print('Seed 123: 98.767%')
print('Seed 777: 99.225%')
print('Mean:     99.001%')
print('Std:       0.229 percentage points')


Seed 42:  99.010%
Seed 123: 98.767%
Seed 777: 99.225%
Mean:     99.001%
Std:       0.229 percentage points


### 33.18 OOD evaluation — what I deliberately did NOT do

PlantDoc was used as an external domain-shift probe. I did **not** fine-tune on PlantDoc, select a checkpoint on PlantDoc, choose ensemble weights on PlantDoc, or change the PlantVillage model based on PlantDoc errors. Doing those things would turn the OOD test into another tuning set.


In [45]:
print('Mapped PlantDoc images: 236')
print('Mapped source classes: 27')
print('EfficientNet accuracy: 23.31%')
print('Ensemble accuracy:     25.00%')


Mapped PlantDoc images: 236
Mapped source classes: 27
EfficientNet accuracy: 23.31%
Ensemble accuracy:     25.00%


## 34. Complete Experimental Decision Log

This table summarizes the project as a sequence of **evidence-driven decisions** rather than a single final training run.

| Evidence observed | Interpretation | Action taken | New evidence |
|---|---|---|---|
| Tiny-set training reached 100% | pipeline can learn | continue to real baseline | baseline train/val curves obtained |
| 1e-3 pilot learned quickly but validation was weaker | LR may be aggressive for long run | baseline full LR = 5e-4 | more conservative long training |
| Dropout 0.60 lagged in short pilot | regularization too strong | keep baseline 0.40 | final scratch baseline established |
| Transfer models reached near-99% | pretrained features highly effective | evaluate ResNet18 and EfficientNet carefully | EfficientNet best single model |
| Historical ensemble = 99.76% | result suspiciously high | run leakage audit | exact/near/same-leaf overlap found |
| 4,956 mapped same-leaf groups crossed splits | image-level split not defensible enough | move to official leaf-safe protocol | mapped train/test overlap = 0 |
| 39 strict dHash pairs remained | residual visual overlap possible | ultra-strict quarantine | strict pairs = 0 |
| Final ensemble still 99.14% | high same-domain result survives audit | test reproducibility | 3 seeds remain near 99% |
| Seed mean 99.001 +/- 0.229 pp | not a single lucky seed | run deeper controls | calibration/robustness/random-label |
| Blur/occlusion caused large degradation | incomplete robustness | document limitation | no deployment claim |
| PlantDoc fell to 23-25% | strong domain shift | revise final claim | PlantVillage benchmark != field performance |


## 35. What Changed in My Understanding During the Project

### At the beginning
My working assumption was: *If train/validation/test file indices do not overlap, the split is clean.*

### After the leakage audit
That statement was too weak for PlantVillage because **different files can show the same physical specimen**. My final standard became: check exact content, perceptual similarity, mapped specimen identity, and keep the final test isolated from model selection.

### At the beginning
My interpretation of 99% was: *The classifier is extremely accurate.*

### At the end
My interpretation became more precise: *The classifier is extremely accurate inside the controlled PlantVillage domain, but this does not automatically transfer to field imagery.*

This change in interpretation is one of the main scientific outcomes of the project.


## 36. Professor Q&A — Concepts Directly Connected to the Code

### Why no Softmax before CrossEntropyLoss?
Because `CrossEntropyLoss` expects raw logits and internally applies the stable log-softmax calculation.

### Why Macro-F1 rather than accuracy for checkpoint selection?
Because all 38 classes receive equal importance, even when class sizes differ.

### Why scheduler on validation loss but checkpoint on Macro-F1?
Loss is a smooth plateau signal; Macro-F1 is a balanced classification metric.

### Why lower learning rate for the backbone?
Pretrained features are already useful, so I update them conservatively. The new classifier needs faster adaptation.

### Did early stopping stop the final runs?
No. It was implemented, but the final selected runs reached their configured maximum epochs first.

### Did leakage explain all of 99.76%?
No. After stricter controls the ensemble still reaches 99.14%.

### Can you guarantee zero leakage?
No. The exact claim is zero **detected** exact, mapped-leaf, and strict dHash<=4 cross-split overlap under the implemented audit.

### Why is PlantDoc much lower?
Domain shift: backgrounds, lighting, framing, scale, and acquisition characteristics differ.

### Why not call 99.225% the best EfficientNet result?
Because selecting the best seed after looking at test results is cherry-picking.

### Does Grad-CAM prove the model uses disease symptoms?
No. It is qualitative supporting evidence only.


## 37. Reproduction Commands

From repository root:

```bat
call .\run_ultrastrict_all.bat
call .\run_full_control_all.bat
```

The first command builds/runs the ultra-strict PlantVillage benchmark. The second performs the deeper validation controls such as calibration, bootstrap CI, random-label sanity, robustness, Grad-CAM, PlantDoc OOD, and seed stability.


## 38. Final Scientific Statement

> **The initial image-level protocol produced an optimistic result and was not sufficient for a defensible evaluation. After exact-duplicate, mapped physical-leaf, and strict perceptual-overlap controls, EfficientNet-B0 still achieved 99.01% and the validation-selected ensemble 99.14% on PlantVillage. Three-seed testing showed that the same-domain result is reproducible, while PlantDoc OOD accuracy of 23-25% demonstrated strong domain dependence. Therefore, the near-99% result is credible as a controlled-domain PlantVillage benchmark, but it should not be interpreted as near-99% real-world field performance.**
